In [5]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from torch.utils.data import DataLoader, Dataset

# Fetch

In [6]:
import kagglehub

In [7]:
temp_dir = Path.cwd() / ".temp"

if temp_dir.parent.name != "model":
    raise RuntimeError()

temp_dir.mkdir(exist_ok=True)
temp_dir

PosixPath('/Users/cube/source/dhbw/ml-digits/model/.temp')

In [8]:
dataset_path = (
    Path(
        kagglehub.dataset_download(
            "metricasecuador/handwritten-digits-version-1-hwd-v1",
            output_dir=str(temp_dir / "dataset"),
        )
    )
    / "HWD-V1"
)
dataset_path

OSError: Could not find a suitable TLS CA certificate bundle, invalid path: /Users/cube/source/dhbw/ml-digits/.venv/lib/python3.14/site-packages/certifi/cacert.pem

# PyTorch

In [ ]:
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


# Dataset

In [ ]:
RESOLUTION = 32

In [ ]:
def dataset_all_split(
    dataset: dict[int, list[Path]], split: float, random_state: int | None = None
):
    """
    Return two split dicts from DATASET_ALL
    """
    train, test = {}, {}

    rng = np.random.default_rng(random_state)

    for digit, items in dataset.items():
        shuffled_items = list(items)
        rng.shuffle(shuffled_items)

        split_idx = int(len(shuffled_items) * split)
        train[digit] = shuffled_items[:split_idx]
        test[digit] = shuffled_items[split_idx:]

    return train, test


def dataset_dict_to_list(dataset_dict):
    return [(item, digit) for digit, items in dataset_dict.items() for item in items]

In [ ]:
DATASET_ITEMS_ALL = {
    digit: list(dataset_path.glob(f"HWD-V1-Standard/{digit}/*.png"))
    for digit in range(10)
}

DATASET_ITEMS_TRAIN, DATASET_ITEMS_TEST = dataset_all_split(
    DATASET_ITEMS_ALL, 0.7, random_state=99
)
DATASET_ITEMS_ALL_LIST, DATASET_ITEMS_TRAIN_LIST, DATASET_ITEMS_TEST_LIST = (
    dataset_dict_to_list(DATASET_ITEMS_ALL),
    dataset_dict_to_list(DATASET_ITEMS_TRAIN),
    dataset_dict_to_list(DATASET_ITEMS_TEST),
)

len(DATASET_ITEMS_TRAIN_LIST), len(DATASET_ITEMS_TEST_LIST)

(105000, 45000)

In [ ]:
class DigitDataset(Dataset):
    def __init__(self, *, items: list[tuple[Path, int]], r: int):
        self.r = r
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        item_path, label = self.items[idx]

        img = Image.open(item_path).convert("L")
        img = ImageOps.invert(img)

        img = img.resize((self.r, self.r))
        img.save(temp_dir / "test_img.png")

        data = np.array(img.get_flattened_data(), dtype=np.float32) / 255
        tensor = torch.tensor(data, dtype=torch.float32)
        tensor = tensor.view(1, self.r, self.r)

        return tensor, label


ds = DigitDataset(items=DATASET_ITEMS_ALL_LIST, r=RESOLUTION)
ds.__getitem__(99999)

(tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]),
 6)

# Model

In [ ]:
@dataclass(kw_only=True)
class TrainResult:
    dataset: Dataset
    model: nn.Module
    epoch: int
    loss: float
    accuracy: float

In [ ]:
def train(
    *, dataset: DigitDataset, model: nn.Module, epochs: int, echo=True
) -> TrainResult:
    loader = DataLoader[DigitDataset](dataset, batch_size=512, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model = model.to(device)

    for epoch in range(epochs):
        total_loss = 0
        correct = 0

        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            Y_pred: torch.Tensor = model(X_batch)
            loss = loss_fn(Y_pred, Y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X_batch.size(0)
            correct += (Y_pred.argmax(1) == Y_batch).sum().item()

        n = len(dataset)
        calc_loss = total_loss / n
        calc_accuracy = correct / n

        if echo:
            print(
                f"Epoch {epoch + 1:2d}  loss={total_loss / n:.4f}  acc={correct / n:.2%}"
            )

    return TrainResult(
        dataset=dataset,
        model=model,
        epoch=epoch,
        loss=calc_loss,
        accuracy=calc_accuracy,
    )

### Very Simple Linear Model

In [57]:
class VerySimpleLinearModel(nn.Module):
    def __init__(self, *, r: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),  # (1, R, R) -> RxR
            nn.Linear(r * r, 10),  # RxR -> 10 digit scores
            nn.Softmax(),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
dataset_train = DigitDataset(items=DATASET_ITEMS_TRAIN_LIST, r=RESOLUTION)

result = train(
    dataset=dataset_train, model=VerySimpleLinearModel(r=RESOLUTION), epochs=60
)

result


/Users/cube/source/dhbw/ml-digits/.venv/lib/python3.14/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch  1  loss=1.7413  acc=85.43%
Epoch  2  loss=1.5503  acc=95.44%
Epoch  3  loss=1.5246  acc=96.35%
Epoch  4  loss=1.5133  acc=96.78%
Epoch  5  loss=1.5067  acc=97.06%
Epoch  6  loss=1.5023  acc=97.24%
Epoch  7  loss=1.4990  acc=97.38%
Epoch  8  loss=1.4965  acc=97.50%
Epoch  9  loss=1.4945  acc=97.63%
Epoch 10  loss=1.4928  acc=97.69%
Epoch 11  loss=1.4914  acc=97.77%
Epoch 12  loss=1.4902  acc=97.81%
Epoch 13  loss=1.4892  acc=97.87%
Epoch 14  loss=1.4883  acc=97.93%
Epoch 15  loss=1.4875  acc=97.99%
Epoch 16  loss=1.4867  acc=98.05%
Epoch 17  loss=1.4860  acc=98.10%
Epoch 18  loss=1.4853  acc=98.14%
Epoch 19  loss=1.4848  acc=98.19%
Epoch 20  loss=1.4842  acc=98.22%
Epoch 21  loss=1.4837  acc=98.25%
Epoch 22  loss=1.4833  acc=98.29%
Epoch 23  loss=1.4828  acc=98.31%
Epoch 24  loss=1.4825  acc=98.34%
Epoch 25  loss=1.4821  acc=98.38%
Epoch 26  loss=1.4817  acc=98.39%
Epoch 27  loss=1.4814  acc=98.42%
Epoch 28  loss=1.4811  acc=98.44%
Epoch 29  loss=1.4808  acc=98.46%
Epoch 30  loss

TrainResult(dataset=<__main__.DigitDataset object at 0x1175dd1d0>, model=VerySimpleLinearModel(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1024, out_features=10, bias=True)
    (2): Softmax(dim=None)
  )
), epoch=59, loss=1.4757756939206805, accuracy=0.9881142857142857)

### Second Model

In [ ]:
class SecondModel(nn.Module):
    def __init__(self, *, r: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(
                1, 8, kernel_size=3, stride=1, padding=1
            ),  # (1, R, R) -> (8, R, R)
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # (8, R, R) -> (8, R/2, R/2)
            nn.Flatten(),  # (8, R/2, R/2) -> (8 * R/2 * R/2)
            nn.Linear(8 * (r // 2) * (r // 2), 100),  # (8 * R/2 * R/2) -> 100
            nn.ReLU(),
            nn.Linear(100, 10),  # (8 * R/2 * R/2) -> 10 digit scores
            nn.Softmax(),
        )

    def forward(self, x):
        return self.net(x)

In [60]:
result2 = train(dataset=dataset_train, model=SecondModel(r=RESOLUTION), epochs=60)
result2

Epoch  1  loss=1.6221  acc=88.37%
Epoch  2  loss=1.4814  acc=98.35%
Epoch  3  loss=1.4749  acc=98.87%
Epoch  4  loss=1.4720  acc=99.10%
Epoch  5  loss=1.4698  acc=99.28%
Epoch  6  loss=1.4686  acc=99.38%
Epoch  7  loss=1.4675  acc=99.49%
Epoch  8  loss=1.4667  acc=99.55%
Epoch  9  loss=1.4658  acc=99.62%
Epoch 10  loss=1.4654  acc=99.65%
Epoch 11  loss=1.4648  acc=99.70%
Epoch 12  loss=1.4644  acc=99.74%
Epoch 13  loss=1.4641  acc=99.77%
Epoch 14  loss=1.4639  acc=99.78%
Epoch 15  loss=1.4637  acc=99.79%
Epoch 16  loss=1.4636  acc=99.80%
Epoch 17  loss=1.4636  acc=99.80%
Epoch 18  loss=1.4632  acc=99.83%
Epoch 19  loss=1.4631  acc=99.84%
Epoch 20  loss=1.4629  acc=99.85%
Epoch 21  loss=1.4629  acc=99.85%
Epoch 22  loss=1.4628  acc=99.85%
Epoch 23  loss=1.4627  acc=99.86%
Epoch 24  loss=1.4631  acc=99.83%
Epoch 25  loss=1.4627  acc=99.87%
Epoch 26  loss=1.4626  acc=99.88%
Epoch 27  loss=1.4625  acc=99.88%
Epoch 28  loss=1.4624  acc=99.88%
Epoch 29  loss=1.4624  acc=99.88%
Epoch 30  loss

TrainResult(dataset=<__main__.DigitDataset object at 0x1175dd1d0>, model=SecondModel(
  (net): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Flatten(start_dim=1, end_dim=-1)
    (4): Linear(in_features=2048, out_features=100, bias=True)
    (5): ReLU()
    (6): Linear(in_features=100, out_features=10, bias=True)
    (7): Softmax(dim=None)
  )
), epoch=59, loss=1.4619583361852737, accuracy=0.9992666666666666)